# Analysis of International Debt Statistics

Exploratory notebook. The final, repeatable version of these steps is `src/run_analysis.py`;
this notebook is where I tried things out first.

Data: World Bank International Debt Statistics (25 indicators, developing countries).

In [ ]:
import sqlite3, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.cleaning import clean
from src.db import build_database

data_file = ROOT / "data/raw/international_debt.csv"
if not data_file.exists():
    data_file = ROOT / "data/raw/international_debt_sample.csv"   # synthetic fallback
print("using", data_file.name)
raw = pd.read_csv(data_file)
raw.head()

## 1. First look at the raw data

In [ ]:
print(raw.shape)
print(raw.dtypes)
raw.isna().sum()

In [ ]:
# Same country + indicator appearing more than once?
raw.duplicated().sum(), raw.duplicated(subset=["country_code", "indicator_code"]).sum()

## 2. Cleaning
Trim spaces, drop exact duplicates and missing values, flag aggregates (regions are not countries).

In [ ]:
df, report = clean(raw)
pd.Series(report)

## 3. Distribution of the debt values
Values range from thousands to tens of billions, so a log scale is needed.

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(np.log10(df.loc[df.debt > 0, "debt"]), bins=40, color="#2f5d8a")
ax.set_xlabel("log10(debt in US$)")
ax.set_ylabel("rows")
plt.show()
df["debt"].describe()

## 4. Load into SQLite and query

In [ ]:
db = build_database(df)
conn = sqlite3.connect(db)
q = lambda sql: pd.read_sql_query(sql, conn)

q("SELECT COUNT(DISTINCT country_name) AS countries FROM countries_debt")

In [ ]:
q('''
SELECT country_name, ROUND(SUM(debt)/1e9, 2) AS total_bn
FROM countries_debt
GROUP BY country_name
ORDER BY total_bn DESC
LIMIT 10
''')

### Double counting check
"Long-term" (DLXF) should equal official + private PPG + private non-guaranteed.
If it does, summing all 25 indicators counts the same money more than once.

In [ ]:
check = q('''
SELECT country_name,
       SUM(CASE WHEN indicator_code = 'DT.AMT.DLXF.CD' THEN debt END) AS long_term,
       SUM(CASE WHEN indicator_code IN ('DT.AMT.OFFT.CD','DT.AMT.PRVT.CD','DT.AMT.DPNG.CD')
                THEN debt END) AS sum_of_parts
FROM countries_debt
GROUP BY country_name
''')
check["ratio"] = check["sum_of_parts"] / check["long_term"]
check["ratio"].describe()

Ratio is ~1 for countries that report all parts, and below 1 where a part is missing.
So: use long-term totals (level 1) for any "total debt" number.

In [ ]:
q('''
SELECT ROUND(SUM(d.debt)/1e6, 2) AS naive_million,
       ROUND(SUM(CASE WHEN i.level = 1 THEN d.debt END)/1e6, 2) AS dedup_million
FROM countries_debt d JOIN indicators i USING (indicator_code)
''')

## 5. Interest burden

In [ ]:
burden = q(open(ROOT / "sql/10_interest_burden.sql").read())
burden

In [ ]:
conn.close()

## Notes / next steps
- Real data has one year only, so no trends over time. Pulling several years with
  `scripts/fetch_worldbank.py` would allow growth rates.
- Debt numbers here are flows, not the stock of debt. Comparing them with GDP would be fairer.